## 1. Method choice and why

My lane is **Lane 2: Refresh / Content Opportunity Scoring**.

For Week 5, I chose a **Decision Tree model**.

I chose this model because it is simple, readable, and easier to explain than a more complex model. My Week 4 baseline was a hand-written rule using visibility, low CTR, and average position. The Decision Tree gives me a simple model to compare against that baseline.

The goal is not to use the most complex model. The goal is to check whether a trained model can beat my Week 4 baseline on the same data and metric.

In [1]:
import os
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

In [3]:
!pip install -q huggingface_hub pyarrow

import os
import json
import numpy as np
import pandas as pd

from google.colab import userdata
from huggingface_hub import list_repo_files, hf_hub_download

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN is not set. Add HF_TOKEN in Colab Secrets first.")

repo_id = "FlyRank/internship-warehouse"
target_table = "fact_content_daily_performance"
target_month = "month=2026-03"

files = list_repo_files(
    repo_id,
    repo_type="dataset",
    token=HF_TOKEN
)

march_files = [
    f for f in files
    if target_table in f
    and target_month in f
    and f.endswith(".parquet")
]

print("March files found:", len(march_files))
print(march_files[:10])

local_files = []

for file_path in march_files:
    local_path = hf_hub_download(
        repo_id=repo_id,
        filename=file_path,
        repo_type="dataset",
        token=HF_TOKEN
    )
    local_files.append(local_path)

df = pd.concat(
    [pd.read_parquet(path) for path in local_files],
    ignore_index=True
)

print("Rows loaded:", len(df))
print("Columns:", len(df.columns))

df.head()

March files found: 1
['fact_content_daily_performance/month=2026-03/data_0.parquet']


fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Rows loaded: 9841378
Columns: 30


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Use the same working data idea from Week 4
work_df = df[df["gsc_data_available"].eq(True)].copy()

# Keep rows with search impressions
work_df = work_df[work_df["gsc_impressions"] > 0].copy()

# Create CTR
work_df["gsc_ctr"] = np.where(
    work_df["gsc_impressions"] > 0,
    work_df["gsc_clicks"] / work_df["gsc_impressions"],
    0
)

# Keep rows with GA4 sessions because our target uses engagement
model_df = work_df[work_df["ga4_sessions"] > 0].copy()

# Target proxy:
# A page may need review if it gets sessions but has zero engaged sessions.
model_df["needs_review_proxy"] = (
    model_df["ga4_engaged_sessions"] == 0
).astype(int)

print("Rows for modeling:", len(model_df))
print("Target distribution:")
print(model_df["needs_review_proxy"].value_counts())
print("Target rate:", round(model_df["needs_review_proxy"].mean(), 3))

Rows for modeling: 361095
Target distribution:
needs_review_proxy
1    335986
0     25109
Name: count, dtype: int64
Target rate: 0.93


## 2. Split design

I used a grouped train/test split by `content_hash_id`.

This is important because the same content page can appear on many dates. If the same content appears in both train and test, the model may look better than it really is.

A grouped split is more honest because it keeps the same content page from leaking across train and test.

In [5]:
feature_cols = [
    "gsc_impressions",
    "gsc_avg_position",
    "gsc_ctr",
    "ga4_pageviews",
    "ga4_sessions",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_ai",
]

X = model_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y = model_df["needs_review_proxy"]
groups = model_df["content_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

test_df = model_df.iloc[test_idx].copy()

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train target rate:", round(y_train.mean(), 3))
print("Test target rate:", round(y_test.mean(), 3))

Train rows: 271508
Test rows: 89587
Train target rate: 0.931
Test target rate: 0.93


In [6]:
# Recreate the Week 4 baseline score on the same test data

test_df["low_ctr_factor"] = 1 - test_df["gsc_ctr"].clip(0, 1)

test_df["position_factor"] = np.where(
    test_df["gsc_avg_position"] <= 20,
    1,
    0.5
)

test_df["baseline_score"] = (
    np.log1p(test_df["gsc_impressions"])
    * test_df["low_ctr_factor"]
    * test_df["position_factor"]
)

test_df[[
    "content_hash_id",
    "report_date",
    "baseline_score",
    "gsc_impressions",
    "gsc_ctr",
    "gsc_avg_position",
    "needs_review_proxy"
]].head()

,content_hash_id,report_date,baseline_score,gsc_impressions,gsc_ctr,gsc_avg_position,needs_review_proxy
14748,content_b1f61fc81b28b2d4,2026-03-01,3.688879,39,0.000000,5.666667,1
14855,content_62673eea26c31c17,2026-03-01,8.094046,3282,0.000305,6.167885,1
15428,content_3c286ded8bd68120,2026-03-01,4.437629,88,0.011364,8.431818,1
17608,content_f8b204c8ce80dad5,2026-03-01,1.098612,2,0.000000,6.500000,1
18916,content_a0e19c582cf792a7,2026-03-01,3.646507,41,0.024390,5.609756,1


In [7]:
model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=1000,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

test_df["model_score"] = model.predict_proba(X_test)[:, 1]

test_df[[
    "content_hash_id",
    "report_date",
    "model_score",
    "baseline_score",
    "needs_review_proxy"
]].head()

,content_hash_id,report_date,model_score,baseline_score,needs_review_proxy
14748,content_b1f61fc81b28b2d4,2026-03-01,0.752162,3.688879,1
14855,content_62673eea26c31c17,2026-03-01,0.977871,8.094046,1
15428,content_3c286ded8bd68120,2026-03-01,0.977871,4.437629,1
17608,content_f8b204c8ce80dad5,2026-03-01,0.938375,1.098612,1
18916,content_a0e19c582cf792a7,2026-03-01,0.977871,3.646507,1


In [8]:
def precision_at_k(scores, y_true, k):
    result_df = pd.DataFrame({
        "score": scores,
        "target": y_true
    }).sort_values("score", ascending=False)

    top_k = result_df.head(k)

    return top_k["target"].mean()


baseline_p50 = precision_at_k(
    test_df["baseline_score"],
    test_df["needs_review_proxy"],
    50
)

model_p50 = precision_at_k(
    test_df["model_score"],
    test_df["needs_review_proxy"],
    50
)

baseline_p100 = precision_at_k(
    test_df["baseline_score"],
    test_df["needs_review_proxy"],
    100
)

model_p100 = precision_at_k(
    test_df["model_score"],
    test_df["needs_review_proxy"],
    100
)

baseline_auc = roc_auc_score(
    test_df["needs_review_proxy"],
    test_df["baseline_score"]
)

model_auc = roc_auc_score(
    test_df["needs_review_proxy"],
    test_df["model_score"]
)

comparison_table = pd.DataFrame({
    "method": ["Week 4 baseline", "Decision Tree model"],
    "precision_at_50": [baseline_p50, model_p50],
    "precision_at_100": [baseline_p100, model_p100],
    "roc_auc": [baseline_auc, model_auc],
})

comparison_table

,method,precision_at_50,precision_at_100,roc_auc
0,Week 4 baseline,0.22,0.35,0.403449
1,Decision Tree model,1.00,1.00,0.864211


## 3. Model vs baseline

The table above compares my Week 4 baseline against the Decision Tree model on the same test split.

The main metric is Precision@50 because the business goal is to create a useful review queue. A content team may only inspect the top recommended pages, so the top of the ranking matters most.

If the model beats the baseline, that means the trained model found useful patterns beyond my hand-written rule.

If the baseline beats the model, that is still useful because it means my simple rule is hard to beat and the model may need better features, a better target, or a different validation design.

In [10]:
feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)

feature_importance

,feature,importance
5,sessions_organic,0.731710
3,ga4_pageviews,0.170794
6,sessions_direct,0.036428
7,sessions_referral,0.027740
4,ga4_sessions,0.023897
2,gsc_ctr,0.006540
0,gsc_impressions,0.002073
1,gsc_avg_position,0.000817
8,sessions_social,0.000000
9,sessions_ai,0.000000


## Feature interpretation

The feature importance table shows which inputs the Decision Tree used most.

This does not prove causation. It only shows which features helped the model make splits.

I should check whether the important features make business sense for my lane. If one feature dominates too much, I should check if it is leaking information or acting as a shortcut.

In [11]:
# Top 100 model picks
test_df["model_rank"] = test_df["model_score"].rank(
    method="first",
    ascending=False
)

test_df["model_top_100"] = test_df["model_rank"] <= 100

false_positives = test_df[
    (test_df["model_top_100"]) &
    (test_df["needs_review_proxy"] == 0)
].sort_values("model_score", ascending=False)

false_negatives = test_df[
    (~test_df["model_top_100"]) &
    (test_df["needs_review_proxy"] == 1)
].sort_values("model_score", ascending=True)

print("False positives in top 100:", len(false_positives))
print("False negatives outside top 100:", len(false_negatives))

false_positives[[
    "content_hash_id",
    "report_date",
    "model_score",
    "gsc_impressions",
    "gsc_ctr",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "needs_review_proxy"
]].head(10)

False positives in top 100: 0
False negatives outside top 100: 83225


,content_hash_id,report_date,model_score,gsc_impressions,gsc_ctr,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,needs_review_proxy


In [12]:
false_negatives[[
    "content_hash_id",
    "report_date",
    "model_score",
    "gsc_impressions",
    "gsc_ctr",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "needs_review_proxy"
]].head(10)

,content_hash_id,report_date,model_score,gsc_impressions,gsc_ctr,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,needs_review_proxy
389675,content_f7219c2c143e0e96,2026-03-02,0.064755,62,0.000000,6.064516,1.0,0.0,1
8298004,content_09c21f570d7d80df,2026-03-27,0.064755,132,0.000000,7.060606,1.0,0.0,1
8959868,content_a9ef090b33d041fd,2026-03-29,0.064755,411,0.000000,12.107056,1.0,0.0,1
3953678,content_1dd45a34666ee684,2026-03-13,0.064755,25,0.000000,8.720000,1.0,0.0,1
2798314,content_b11eb17007c1cbd4,2026-03-09,0.064755,20,0.000000,4.150000,1.0,0.0,1
3953141,content_7b8ac34c47437779,2026-03-13,0.064755,95,0.000000,23.800000,1.0,0.0,1
8705219,content_ea30e634462add50,2026-03-27,0.064755,22,0.000000,16.863636,1.0,0.0,1
7912304,content_7e978212c9d6c6e8,2026-03-24,0.064755,573,0.000000,2.349040,1.0,0.0,1
4757656,content_a602897cef7c07cf,2026-03-15,0.064755,426,0.000000,4.281690,1.0,0.0,1
7911970,content_82e35c4845e6c391,2026-03-24,0.064755,3465,0.000289,35.740260,1.0,0.0,1


## 4. Errors and interpretation

The model can make two main types of mistakes.

False positives are pages the model ranks highly for review, but the target says they are not review candidates. These may be pages that look weak from search or traffic signals but still have engagement.

False negatives are pages the model ranks too low, but the target says they may need review. These may be pages where the weak signal is not captured by my current features.

The errors show that this model should be treated as decision support, not automatic truth. A human still needs to inspect the page, query intent, page purpose, and business context before making changes.

In [13]:
output_dir = "work/outputs"
os.makedirs(output_dir, exist_ok=True)

w05_metrics = {
    "lane": "Lane 2 - Refresh / Content Opportunity Scoring",
    "model": "DecisionTreeClassifier",
    "split": "GroupShuffleSplit by content_hash_id",
    "target_proxy": "needs_review_proxy = ga4_sessions > 0 and ga4_engaged_sessions == 0",
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "baseline_precision_at_50": float(baseline_p50),
    "model_precision_at_50": float(model_p50),
    "baseline_precision_at_100": float(baseline_p100),
    "model_precision_at_100": float(model_p100),
    "baseline_roc_auc": float(baseline_auc),
    "model_roc_auc": float(model_auc),
}

json_path = os.path.join(output_dir, "w05_model_metrics.json")

with open(json_path, "w") as f:
    json.dump(w05_metrics, f, indent=2)

print("Wrote:", json_path)
w05_metrics

Wrote: work/outputs/w05_model_metrics.json


{'lane': 'Lane 2 - Refresh / Content Opportunity Scoring',
 'model': 'DecisionTreeClassifier',
 'split': 'GroupShuffleSplit by content_hash_id',
 'target_proxy': 'needs_review_proxy = ga4_sessions > 0 and ga4_engaged_sessions == 0',
 'train_rows': 271508,
 'test_rows': 89587,
 'baseline_precision_at_50': 0.22,
 'model_precision_at_50': 1.0,
 'baseline_precision_at_100': 0.35,
 'model_precision_at_100': 1.0,
 'baseline_roc_auc': 0.40344863167249334,
 'model_roc_auc': 0.864211223421927}

## 5. Self-check

- [x] I used my locked lane: Lane 2, Refresh / Content Opportunity Scoring.
- [x] I explained why I chose the method.
- [x] I used a grouped train/test split by `content_hash_id`.
- [x] I recreated my Week 4 baseline on the same test data.
- [x] I trained a Decision Tree model.
- [x] I compared the model and baseline on the same split.
- [x] I reported Precision@50, Precision@100, and ROC AUC.
- [x] I interpreted the model features.
- [x] I reviewed errors and limitations.
- [x] I avoided using the target column as an input feature.
- [x] I did not reward complexity alone.